# The Dolinar Receiver

Telling apart two coherent states $|{+}\alpha\rangle$ and $|{-}\alpha\rangle$ with
nothing but a displacement, one single-photon detector, and feedback.

**Scope of this notebook.** Only the Dolinar part is developed here. The Kennedy
and homodyne curves are kept because the comparison plot needs them, but they are
documented in a separate notebook.

| Part | Question | Where it lands |
|---|---|---|
| 1 | Why is perfect discrimination impossible, and why is Kennedy not enough? | feedback is *necessary* |
| 2 | How do we compress "everything seen so far" into one number? | $\lambda' = \lambda + \ln L_+ - \ln L_-$ |
| 3 | What do we do with that number? | $\beta^* = -a/m$ (next iteration) |

Part 3 (the control law, the invariant $Q$, and the Bellman recursion) is only
*used* below, not derived: the derivation comes in the next pass.

In [ ]:
# this is the part where we import
import numpy as np
import matplotlib.pyplot as plt
import qutip as q
from scipy.special import erfc
from scipy.optimize import brentq

---

# Part 1. The problem

## 1.1 Two signals that cannot be separated

A laser pulse is a **coherent state** $|\alpha\rangle$, with $|\alpha|^2$ the mean
photon number. Expanded in the Fock basis,

$$|\alpha\rangle = e^{-|\alpha|^2/2}\sum_{n=0}^{\infty}\frac{\alpha^n}{\sqrt{n!}}\,|n\rangle
\quad\Longrightarrow\quad
P(n) = e^{-|\alpha|^2}\frac{(|\alpha|^2)^n}{n!}$$

Counting photons gives a **Poisson distribution** with mean $|\alpha|^2$. The single
term that carries the whole notebook is $n=0$:

$$P(0) = e^{-|\alpha|^2} \tag{1.1}$$

Encode bit 0 as $|{+}\alpha\rangle$ and bit 1 as $|{-}\alpha\rangle$ (BPSK). The two
states are **not orthogonal**:

$$\langle -\alpha|\alpha\rangle = e^{-2|\alpha|^2},
\qquad |\langle -\alpha|\alpha\rangle|^2 = e^{-4|\alpha|^2} \tag{1.2}$$

Non-orthogonal states cannot be told apart perfectly by *any* measurement. This is
not a hardware limitation, so zero error probability is off the table from the start;
the only question left is what the minimum is.

## 1.2 The best achievable: Helstrom bound

For two pure states received with priors $p$ and $1-p$,

$$P_e^{\text{Hel}} = \frac{1}{2}\Big(1 - \sqrt{1 - 4p(1-p)\,|\langle\psi_+|\psi_-\rangle|^2}\Big)
\tag{1.3}$$

With $p = 1/2$ and the overlap (1.2),

$$P_e^{\text{Hel}} = \frac{1}{2}\Big(1 - \sqrt{1 - e^{-4|\alpha|^2}}\Big) \tag{1.4}$$

This is a lower bound over **all** quantum measurements. What it does *not* give is a
recipe: the optimal measurement is an abstract projection, not an optical bench. The
point of the Dolinar receiver is that a displacement, one detector, and feedback
reach this bound exactly.

## 1.3 The Kennedy receiver: erase, then listen

A displacement $D(\beta)$ — physically, mixing the signal with a strong local
oscillator on an unbalanced beam splitter — just adds amplitudes:

$$D(\beta)|\gamma\rangle \;\propto\; |\gamma + \beta\rangle$$

Set $\beta = -\alpha$:

* under $H_+$: $|{+}\alpha\rangle \to |0\rangle$ — **vacuum**, no photons at all;
* under $H_-$: $|{-}\alpha\rangle \to |{-}2\alpha\rangle$ — mean photon number $4|\alpha|^2$.

A single-photon detector (SPD) cannot count, it only reports **click / no click**, so
the decision rule is forced: *click $\to$ $-\alpha$, silence $\to$ $+\alpha$.* Vacuum
never clicks, so the only error is a silent $H_-$, and by (1.1)

$$P_e^{\text{Ken}} = \tfrac12\,e^{-4|\alpha|^2} \tag{1.5}$$

Compared with (1.4), which behaves as $\tfrac14 e^{-4|\alpha|^2}$ at large $|\alpha|$,
**Kennedy is off by exactly a factor of 2.** Killing that factor is the rest of the story.

## 1.4 Chopping into $N$ slots changes nothing

Split the pulse into $N$ time slots. Energy conservation fixes the per-slot amplitude:

$$N a^2 = \alpha^2 \quad\Longrightarrow\quad a = \frac{\alpha}{\sqrt N} \tag{1.6}$$

Null every slot with $\beta = -a$ and stop at the first click. Under $H_+$ every slot
is vacuum, so no error. Under $H_-$ each slot is $|{-}2a\rangle$ with mean $4a^2$, and
an error needs **all** slots silent:

$$P(\text{all silent}\mid H_-) = \big(e^{-4a^2}\big)^N = e^{-4Na^2} = e^{-4\alpha^2}
\quad\Longrightarrow\quad P_e = \tfrac12 e^{-4\alpha^2} = P_e^{\text{Ken}} \tag{1.7}$$

**Slicing alone buys exactly nothing** — the exponential factorises back into itself.
This is a milestone, not a disappointment: it isolates what is actually missing.

## 1.5 What has to change

Nothing in the (1.7) calculation used the fact that slot 3 clicked when choosing what
to do in slot 4. Every slot used $\beta = -a$, blindly.

Full nulling also has a hidden cost. Since $H_+$ clicks with probability exactly $0$,
a single click makes $H_-$ **certain and irreversible** — no number of remaining slots
can undo it. Dolinar's move is:

> displace each slot not by $-a$, but by $\beta_k$ **computed from everything observed
> so far**.

That needs two things:

1. a way to summarise "everything observed so far" as a number $\to$ **Part 2** ($p$, $\lambda$, $m$)
2. a rule turning that number into $\beta_k$ $\to$ **Part 3** ($\beta^* = -a/m$)

---

# Part 2. $p$ — the mathematics of belief

## 2.1 What we want

$$p \;\equiv\; P(H_+ \mid \text{everything observed so far})$$

Physics hands us the opposite direction: Poisson gives $P(\text{click}\mid H_\pm)$, not
$P(H_\pm\mid\text{click})$. Bayes' theorem is the device that reverses the arrow.

Note what kind of probability this is. $|\langle\phi|\psi\rangle|^2$ is objective;
$p$ is not. "This pulse is $+\alpha$ with probability 0.7" describes the **state of
information in the receiver**, not the pulse — the pulse is already one or the other.
Two receivers with different click histories legitimately hold different $p$.
Quantum mechanics enters this problem exactly once, through (1.1); everything after
the likelihoods is ordinary classical probability.

## 2.2 The update

With $A = H_+$, $B = o_k$ (this slot's outcome), $C = o_{<k}$ (the history),

$$P(H_+\mid o_k, o_{<k}) = \frac{P(o_k\mid H_+, o_{<k})\;P(H_+\mid o_{<k})}{P(o_k\mid o_{<k})}
\tag{2.4}$$

| symbol | meaning | name |
|---|---|---|
| $P(H_+\mid o_k, o_{<k}) = p_k$ | belief after this slot | posterior |
| $P(H_+\mid o_{<k}) = p_{k-1}$ | belief before this slot | prior |
| $P(o_k\mid H_\pm, o_{<k}) = L_\pm^{(k)}$ | how well each hypothesis explains this outcome | likelihood |

The second row is where the recursion comes from: **yesterday's posterior is today's
prior** is not an extra assumption, it is the same symbol used twice.

The denominator is not computed separately — $\{H_+,H_-\}$ is an exclusive and
exhaustive split, so

$$P(o_k\mid o_{<k}) = p_{k-1}L_+^{(k)} + (1-p_{k-1})L_-^{(k)} \tag{2.5}$$

This quantity is the **evidence**: the receiver's own prediction of what it is about to
see. Putting (2.4) and (2.5) together,

$$p_k = \frac{p_{k-1}L_+^{(k)}}{p_{k-1}L_+^{(k)} + (1-p_{k-1})L_-^{(k)}} \tag{2.6}$$

The outcome variable takes two values, $o \in \{0,1\}$: $0$ = silence, $1$ = click.
**Silence is a genuine observation** — its evidence is not zero, and since most slots
are silent in normal operation, most of the accumulated evidence comes from silence.

## 2.3 The likelihoods — where the physics enters

The history influences this slot through exactly one channel: it fixes $\beta_k$.
Conditioning on the history therefore makes the slot an independent one-shot
experiment with a known displacement. After displacement the state is
$|{\pm}a + \beta_k\rangle$, so by (1.1) the photon number is Poisson with

$$\mu_\pm^{(k)} = (\pm a + \beta_k)^2 \tag{2.7}$$

and since the SPD only distinguishes zero from nonzero,

$$L_\pm^{(k)} = \begin{cases} e^{-\mu_\pm^{(k)}} & o = 0 \;(\text{silence}) \\[4pt]
1 - e^{-\mu_\pm^{(k)}} & o = 1 \;(\text{click})\end{cases} \tag{2.8}$$

The formula is literally identical for the first slot and the hundredth; the whole
history has been absorbed into $\beta_k$.

Two warnings that matter when reading the code. $L_+$ and $L_-$ are probabilities
under *different conditions*, so they do not sum to $1$ across hypotheses — only across
outcomes, $L_\pm(0) + L_\pm(1) = 1$. And the receiver does not know which hypothesis is
true, so it **runs both counterfactuals in parallel**: that is why `mu_p` and `mu_m` are
always computed side by side.

## 2.4 Odds and log-odds

Equation (2.6) computes, but hides the structure. In odds $O \equiv p/(1-p)$ the
normalisation cancels:

$$O_k = O_{k-1}\cdot\frac{L_+^{(k)}}{L_-^{(k)}}
\qquad\Longleftrightarrow\qquad
\text{new odds} = \text{old odds} \times \text{likelihood ratio} \tag{2.9}$$

Taking a log turns the product into a sum:

$$\lambda \equiv \ln\frac{p}{1-p}
\quad\Longrightarrow\quad
\lambda_k = \lambda_{k-1} + \ln L_+^{(k)} - \ln L_-^{(k)} \tag{2.10}$$

$$p = \frac{1}{1+e^{-\lambda}} \tag{2.11}$$

Each observation contributes a signed amount of evidence and they simply accumulate.
Note that **independence is never assumed**: with feedback the outcomes are *not*
conditionally independent ($o_1$ moves $\beta_2$, which moves the distribution of
$o_2$). What is used is the chain rule, which holds unconditionally — and having
$o_{<k}$ in the conditioning is exactly what pins down $\beta_k$.

**Sufficient statistic.** Everything the click history says about the hypothesis is
compressed into the single number $\lambda$; the history itself need not be stored.
In control language $\lambda$ is the *belief state*, and two consequences follow:

* the problem becomes **Markov** in $\lambda$, so backward induction solves it exactly;
* for an RL formulation, feeding $\lambda$ alone is enough — no recurrence over history.

## 2.5 $m = 2p-1$

$$m \equiv 2p - 1 = P(H_+\mid\text{hist}) - P(H_-\mid\text{hist}) \tag{2.13}$$

A signed confidence: sign = which hypothesis, magnitude = how sure. No new
information, just a coordinate with its neutral point at $0$ instead of $0.5$.
Three reasons it is the right one:

**(a) It reads off the error probability.** Stopping now and guessing costs
$\min(p, 1-p)$, i.e.

$$P_e^{\text{now}} = \frac{1 - |m|}{2} \tag{2.14}$$

**(b) It diagonalises the symmetry.** Swapping the labels $+\leftrightarrow-$ sends
$p \to 1-p$, i.e. $m \to -m$ and $\beta \to -\beta$; so $\beta^*$ is odd in $m$ and the
value function is even.

**(c) $1 - m^2 = 4p(1-p)$** is exactly the combination appearing in the Helstrom
bound (1.3) — this is what Part 3 builds on.

From (2.11) and (2.13),

$$m = \frac{1-e^{-\lambda}}{1+e^{-\lambda}} = \tanh(\lambda/2) \tag{2.15}$$

The $\tanh$ is not imported from physics; it falls out of inverting the definition of $\lambda$.

## Part 2 in one line — three coordinates for one fact

$$\lambda = \ln\frac{p}{1-p}, \qquad p = \frac{1}{1+e^{-\lambda}},
\qquad m = 2p-1 = \tanh(\lambda/2)$$

* $\lambda$ — **for updating**: additive, numerically safe.
* $m$ — **for control**: what Part 3 feeds into $\beta^* = -a/m$.
* $p$ — **for reading**: the human-legible one.

---

## Part 2 $\to$ code: where each piece lives

The cell below is the **theory-policy Dolinar receiver**: backward induction in which
$\beta$ is not searched for but evaluated from $\beta^* = -a/m$ (Part 3). Every line of
`dolinar_update_theory` is one equation of Part 2, in order.

| Concept | Equation | Code |
|---|---|---|
| per-slot amplitude, energy conserved | (1.6) $a=\alpha/\sqrt N$ | `a = alpha / np.sqrt(N)` |
| belief $p$ from log-odds | (2.11) | `p = _sigmoid(lam)` |
| belief axis $\lambda$ (the sufficient statistic) | (2.10), §2.4 | `lam = _lam_grid(alpha, n_lam)` |
| mean photon number after displacement | (2.7) $\mu_\pm=(\pm a+\beta)^2$ | `mu_p`, `mu_m` |
| likelihood of the SPD outcome | (2.8) | `L_analytic(mu, o)` $\to$ `L_p`, `L_m` |
| both counterfactuals run in parallel | §2.3 | `L_p` and `L_m` computed side by side |
| evidence / predicted outcome probability | (2.5) $pL_++(1-p)L_-$ | the weight in `tot = tot + (...) * ...` |
| Bayes update in log-odds | (2.10) $\lambda'=\lambda+\ln L_+-\ln L_-$ | `nxt = lam + np.log(L_p) - np.log(L_m)` |
| silence *is* an observation | §2.2 | the `for o in (0, 1)` loop, `o = 0` branch |
| cost of guessing now | (2.14) $\min(p,1-p)=\tfrac{1-\lvert m\rvert}{2}$ | `V = np.minimum(p, 1 - p)` (the $k=0$ boundary) |
| confidence $m$ | (2.15) $m=\tanh(\lambda/2)$ | `m = np.tanh(lam / 2)` in `beta_theory` |
| control law (Part 3) | $\beta^*=-a/m$ | `np.clip(-a / m, -bmax, bmax)` |
| prior is 50:50 | $\lambda_0 = 0$ | `np.interp(0.0, lam, V)` |

**Two grids, two very different roles.** The $\lambda$ grid is a pure numerical device:
refine it and the answer must not move (it is a converged discretisation of the belief
axis). The $\beta$ grid in the DP cell is an *algorithmic choice* — dropping it in
favour of the closed-form control genuinely changes the answer at finite $N$.

**Dependencies.** This cell calls `L_analytic`, `_sigmoid`, `_lam_grid` (defined in the
DP cell further down) and `beta_opt` (defined in the Kennedy cell). Run those cells
first, or move them above this one.

In [ ]:
def error_ke(N,alpha_ke,beta_ke=None):
    if beta_ke == None:
        beta_ke=alpha_ke
    a_p=q.coherent(N,alpha_ke) #a_p is positive alpha
    a_n=q.coherent(N,-alpha_ke) #a_n is negative alpha
    #displacement operator
    D_beta = q.displace(N, beta_ke)
    a_dp= D_beta*a_p
    a_dn= D_beta*a_n
    p_dp = abs(a_dp.full()[0][0])**2
    p_dn = abs(a_dn.full()[0][0])**2
    
    return p_dp / 2 + (1 - p_dn) / 2
    

def beta_opt(a):
    """Single-shot optimal displacement: the root of 4ab = ln((b+a)/(b-a)).
    Used as the cap bmax in the theory policy."""
    if a > 3.0:                       # b* - a < 1 ULP, not resolvable in double precision
        return a
    g = lambda b: 4*a*b - np.log((b + a)/(b - a))
    lo = np.nextafter(a, np.inf)      # next representable double above a
    return brentq(g, lo, 3*a + 1)

In [ ]:
# ================= Theory policy (beta* = -a/m) =================
# Same backward induction as the DP cell, except beta is not picked out of a
# candidate grid: it is evaluated straight from the closed-form control law.
# Put dolinar_update and dolinar_update_theory side by side and the only
# difference is the argmin line.
# optimal version of dolinar (tuned for small numbers)

# ================= Likelihood =================
def L_analytic(mu, o):
    """Likelihood of outcome o for mean photon number mu, eq. (2.8).
    o = 0 silence -> exp(-mu),  o = 1 click -> 1 - exp(-mu).
    expm1 keeps precision when mu is tiny (1 - e^-mu would cancel to noise)."""
    return (np.exp(-mu), -np.expm1(-mu))[o]


def _auto_n_lam(alpha, N, per_step=10, hard_cap=400001):
    """Grid points needed to split one slot's evidence step (~4a^2) into
    per_step cells. Any coarser and the update is buried in the grid spacing,
    so the value silently comes out wrong -- no error, just a plausible number."""
    a2 = alpha**2 / N
    lam_max = max(12.0, 6.0 * alpha**2 + 12.0)
    n = int(2 * lam_max * per_step / (4 * a2)) + 1
    return int(np.clip(n, 601, hard_cap))


def beta_theory(lam, a, bmax=np.inf):
    """Control law beta* = -a/m with m = tanh(lam/2), eq. (2.15).
    bmax clips it. m = 0 makes -a/m blow up, and without the cap the receiver
    locks up (see dolinar_error_theory)."""
    m = np.tanh(lam / 2)                       # confidence, eq. (2.15)
    m = np.where(np.abs(m) < 1e-12, 1e-12, m)  # keep the division finite at m = 0
    return np.clip(-a / m, -bmax, bmax)        # clip handles the sign on its own:
                                               # m > 0 -> -a/m < 0 -> cut at -bmax


def dolinar_update_theory(V, lam, a, beta):
    """One Bellman step, minus the minimisation. beta is (n_lam,) -- the control
    is already fixed by the belief itself.

        V_k(lam) = sum_o  P(o) * V_{k-1}(lam')      <- no min_beta here

    Read line by line, this is Part 2 in order:
        p      = sigmoid(lam)              (2.11)  belief in H+
        mu_pm  = (+-a + beta)^2            (2.7)   mean photons after displacement
        L_pm   = L_analytic(mu_pm, o)      (2.8)   SPD likelihood
        P(o)   = p L_+ + (1-p) L_-         (2.5)   evidence
        lam'   = lam + ln L_+ - ln L_-     (2.10)  Bayes update
    """
    p = _sigmoid(lam)                          # (n_lam,)  prior belief this slot, (2.11)
    mu_p = ( a + beta) ** 2 + 1e-300           # (n_lam,)  mean photon number under H+, (2.7)
    mu_m = (-a + beta) ** 2 + 1e-300           # (n_lam,)  ... and under H-. 1e-300 keeps
                                               #           log() finite at perfect nulling

    tot = 0.0                                  # accumulates an (n_lam,) array
    for o in (0, 1):                           # o = 0 silence, o = 1 click -- silence is
                                               # an observation too, and it carries most
                                               # of the evidence in normal operation
        L_p = L_analytic(mu_p, o)              # P(o | H+, history), (2.8)
        L_m = L_analytic(mu_m, o)              # P(o | H-, history) -- both counterfactuals
                                               # are run in parallel; only one is real
        nxt = lam + np.log(L_p + 1e-300) - np.log(L_m + 1e-300)   # posterior lam', (2.10)
        tot = tot + (p * L_p + (1 - p) * L_m) * np.interp(nxt, lam, V)
        #            \_______ evidence P(o|hist), (2.5) _______/   \__ V_{k-1} read at lam' __/
        # note: V is interpolated, not log V -- V is convex, so linear interpolation
        # errs on the safe side, whereas log V is concave and would push the answer
        # below the Helstrom bound (an impossible value = a bug)

    return tot, beta                           # the DP version takes an argmin right here


def dolinar_error_theory(alpha, N, n_lam=None, cap=True,
                         per_step=10, return_policy=False):
    """Error probability of the closed-form policy. Same skeleton as dolinar_error_sopt.

    With cap=False, beta diverges at m = 0 (lam = 0, i.e. the very first slot with a
    50:50 prior). Then mu >> 1 for both hypotheses, so L_+(1) = L_-(1) = 1, the
    likelihood ratio is 1, lam never moves, and the receiver locks at Pe = 0.5.
    Worth running once to see why the divergence matters.
    """
    if alpha <= 0:
        return (0.5, None) if return_policy else 0.5
    if n_lam is None:
        n_lam = _auto_n_lam(alpha, N, per_step)

    a    = alpha / np.sqrt(N)                  # per-slot amplitude, energy conserved: (1.6)
    lam  = _lam_grid(alpha, n_lam)             # belief axis (the sufficient statistic)
    bmax = beta_opt(a) if cap else np.inf      # single-shot optimal displacement, from
                                               # the Kennedy cell: a cutoff with no free
                                               # parameter, and it vanishes as N grows
    beta = beta_theory(lam, a, bmax)           # control: formula instead of a grid

    p = _sigmoid(lam)
    V = np.minimum(p, 1 - p)                   # k = 0 boundary: just guess, (2.14)
    policy = []
    for _ in range(N):                         # 1 slot remaining -> N slots remaining
        V, beta_star = dolinar_update_theory(V, lam, a, beta)
        policy.append(beta_star)               # policy[k-1] = policy with k slots left

    pe = float(np.interp(0.0, lam, V))         # 50:50 prior means lam = 0
    return (pe, (lam, policy)) if return_policy else pe

In [ ]:
alphas=np.linspace(0,2,200)

# 2. Ideal Kennedy
P_kennedy = 0.5 * np.exp(-4 * np.abs(alphas)**2)

# 3. Helstrom bound
overlap_squared = np.exp(-4 * np.abs(alphas)**2)
P_helstrom = 0.5 * (1 - np.sqrt(1 - overlap_squared))

#optimal kennedy
P_kennedy_opt=[error_ke(30,i,beta_opt(i)) for i in alphas]

#homodyne
P_hom_theory = 0.5 * erfc(np.sqrt(2) * np.abs(alphas))

#dolinar receiver -- N = 400 slots; the finite-N control law needs many slots
#before the cap at m = 0 stops mattering
P_dolinar = np.array([dolinar_error_theory(al, 400) for al in alphas])

In [ ]:
# --- Simplest Plot Structure ---
fig, ax = plt.subplots(figsize=(6, 4))

# thick and semi-transparent, so it sits in the background
ax.plot(alphas, P_kennedy, label="Ideal Kennedy", linewidth=5, alpha=0.4)

# thin dashed line on top: shows the two curves lie exactly on each other
ax.plot(alphas,P_kennedy_opt , '--', label="optimal kennedy", color='red')

# Helstrom bound
ax.plot(alphas, P_helstrom, '-.', label="Helstrom bound", color='black')

# homodyne
ax.plot(alphas, P_hom_theory, '-.', label="Homodyne", color='orange')

# Dolinar receiver
ax.plot(alphas, P_dolinar, label="Dolinar (N=400)", color='green', lw=2)

#ax.set_yscale('log')   # without this the three curves look glued together
ax.set_xlabel(r"$|\alpha|$")
ax.set_ylabel("Error Probability")
ax.grid(True)
ax.legend()

plt.show()

---

## The exact DP on a grid

Same backward induction, but $\beta$ is chosen by brute force over a candidate grid
instead of from the formula:

$$V_0(\lambda) = \min(p,\,1-p), \qquad
V_k(\lambda) = \min_\beta \sum_o \big[pL_+ + (1-p)L_-\big]\,
V_{k-1}\Big(\lambda+\ln\tfrac{L_+}{L_-}\Big), \qquad P_e = V_N(0)$$

The only difference from `dolinar_update_theory` is the `argmin` line, and the shape of
`beta`: here it is an independent axis `(n_beta,)`, so `tot` is an `(n_lam, n_beta)`
table and each belief row picks its own best control. In the theory version `beta` is
`(n_lam,)` — the control axis has been absorbed into the belief axis.

The $\beta$ candidates cannot be laid out carelessly: $x = \beta/a = -1$ (perfect
nulling) must sit *exactly* on the grid, otherwise a spurious $\mu_+ \sim 10^{-5}$
survives and swamps the true answer at large $\alpha$. Hence $s = |\beta| - a$ is
spaced geometrically **starting from 0**.

In [ ]:
# optimal version of dolinar (tuned for small numbers)

# ================= Likelihood =================
def L_analytic(mu, o):
    """Likelihood of outcome o for mean photon number mu, eq. (2.8).
    o = 0 silence -> exp(-mu),  o = 1 click -> 1 - exp(-mu).
    expm1 keeps precision when mu is tiny (1 - e^-mu would cancel to noise)."""
    return (np.exp(-mu), -np.expm1(-mu))[o]

# ================= Exact DP on a grid =================
def _beta_grid(a, n_s=150):
    """Displacement candidates: |beta| >= a, with s = |beta| - a spaced
    geometrically from 0. s = 0 is perfect nulling (x = -1), and that point has
    to be hit exactly at large alpha or a spurious mu_+ swamps the true answer."""
    s = np.concatenate([[0.0], np.geomspace(1e-9, 2.5, n_s)])
    return np.concatenate([-(a + s), (a + s)])

def _sigmoid(x):
    """p = 1 / (1 + e^-lam), eq. (2.11). Written this way to avoid overflowing
    on e^lam for large positive lam."""
    return 1.0 / (1.0 + np.exp(-x))


def _lam_grid(alpha, n_lam=601):
    """Belief axis. lam_max must cover how confident the receiver gets at large
    alpha. Pure numerics: refine it and the answer must not move."""
    lam_max = max(12.0, 6.0 * alpha ** 2 + 12.0)
    return np.linspace(-lam_max, lam_max, n_lam)


def dolinar_update(V, lam, a, beta):
    """One Bellman step. Takes V = V_{k-1}, returns (V_k, the optimal beta(lam)).

        V_k(lam) = min_beta  sum_o  P(o) * V_{k-1}(lam')
        P(o)  = p L+ + (1-p) L-          evidence: probability of seeing o, (2.5)
        lam'  = lam + log L+ - log L-    Bayes update in log-odds, (2.10)

    Same content as dolinar_update_theory, one axis wider: here beta is an
    independent axis, so tot is a 2-D table and every belief row picks its own
    minimum. That last argmin is the whole difference between the two versions.
    """
    p = _sigmoid(lam)
    P, LAM = p[:, None], lam[:, None]          # belief axis   (n_lam, 1)
    B = beta[None, :]                          # control axis  (1, n_beta)
    mu_p = ( a + B) ** 2 + 1e-300              # (n_lam, n_beta) by broadcasting, (2.7)
    mu_m = (-a + B) ** 2 + 1e-300

    tot = 0.0                                  # (n_lam, n_beta) table
    for o in (0, 1):                           # o = 0 silence, o = 1 click
        L_p = L_analytic(mu_p, o)              # P(o | H+), (2.8)
        L_m = L_analytic(mu_m, o)              # P(o | H-)
        nxt = LAM + np.log(L_p + 1e-300) - np.log(L_m + 1e-300)   # lam', (2.10)
        tot = tot + (P * L_p + (1 - P) * L_m) * np.interp(nxt, lam, V)

    j = tot.argmin(axis=1)                     # one optimal control per belief
    return tot[np.arange(lam.size), j], beta[j]


def dolinar_error_sopt(alpha, N, n_lam=601, n_s=150, return_policy=False):
    """Error probability straight from backward induction.
    V_k(lam) = lowest error probability reachable with k slots left."""
    if alpha <= 0:
        return (0.5, None) if return_policy else 0.5

    a    = alpha / np.sqrt(N)                  # per-slot amplitude (energy conserved), (1.6)
    lam  = _lam_grid(alpha, n_lam)             # belief axis
    beta = _beta_grid(a, n_s)                  # control axis

    p = _sigmoid(lam)
    V = np.minimum(p, 1 - p)                   # k = 0 boundary: just guess, (2.14)
    policy = []
    for _ in range(N):                         # 1 slot remaining -> N slots remaining
        V, beta_star = dolinar_update(V, lam, a, beta)
        policy.append(beta_star)               # policy[k-1] = policy with k slots left

    pe = float(np.interp(0.0, lam, V))         # 50:50 prior means lam = 0
    return (pe, (lam, policy)) if return_policy else pe